In [ ]:
import os

# Working directory must contain AlphaSimPy.py for imports
os.chdir(r"/Users/mtwatson/Library/CloudStorage/Box-Box/Projects/AI agent for breeding/Endpoint 2 agent")


# Vanvanhossou Breeding Program - AlphaSimPy Notebook

This notebook converts the provided BRAID breeding program abstraction into a tutorial-style AlphaSimPy simulation.

The BRAID program describes:
- two founder sources: **local African taurine** and **exotic Asian indicine**
- independent advancement from generation 0 to generation 20
- **phenotypic selection** in the local population for **tick count incidence**
- **genomic selection** in the exotic population for **body weight**
- **crossbreeding** between selected local and exotic groups from generations 21 to 40

**Package**: AlphaSimPy  
**Source**: BRAID abstraction  
**Program horizon**: 40 generations


## Assumptions Used

The BRAID abstraction intentionally leaves several numeric details unspecified. To make the notebook executable, the following explicit assumptions are used:

1. Each founder source starts with a fixed number of individuals.
2. The two traits are modeled as additive traits with moderate heritability.
3. The local population is selected on **phenotype** for tick count incidence. Since lower tick count is favorable, selection is applied to the lowest phenotypic values.
4. The exotic population is selected on **breeding value** for body weight. Because the BRAID file specifies a placeholder genomic prediction model, this notebook uses **true genetic value as an EBV proxy** to represent genomic selection logic in a deterministic and runnable way.
5. Hybrid production from generations 21 to 40 is represented as repeated crossing between the selected local and selected exotic populations, with summary statistics tracked each generation.
6. Where AlphaSimPy function support can differ across installations, the code uses standard AlphaSimR-style camelCase function names and keeps the workflow simple and transparent.

These assumptions are documented so the notebook remains faithful to the BRAID abstraction while still being runnable.


## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from AlphaSimPy import runMacs, SimParam, newPop, randCross, setPheno, selectInd, meanG, varG

print("AlphaSimPy BRAID Conversion Notebook")
print("Libraries imported successfully.")

## Global Parameters

Set simulation parameters based on the BRAID abstraction and explicit assumptions for missing values.


In [ ]:
# Program settings from BRAID
program_name = "Vanvanhossou"
n_generations_total = 40
n_advance = 20
n_hybrid_phase = 20

# Genome settings from BRAID
ploidy = 2
n_chr = 30
n_qtl_per_trait = 100
n_snp = 200

# Trait settings from BRAID
h2_tick = 0.30
h2_weight = 0.30

# Assumed founder and breeding sizes
n_founders_per_pool = 100
n_crosses_per_generation = 50
selection_intensity = 0.50
n_selected = int(n_founders_per_pool * selection_intensity)
n_hybrid_crosses = 50

# Error variances chosen to roughly reflect moderate heritability
varE_tick = 1.0
varE_weight = 1.0

print("Simulation parameters:")
print(f"  Program: {program_name}")
print(f"  Total generations: {n_generations_total}")
print(f"  Advancement generations: {n_advance}")
print(f"  Hybrid generations: {n_hybrid_phase}")
print(f"  Chromosomes: {n_chr}")
print(f"  QTL per trait: {n_qtl_per_trait}")
print(f"  SNP per chromosome: {n_snp}")
print(f"  Founders per pool: {n_founders_per_pool}")
print(f"  Crosses per generation: {n_crosses_per_generation}")
print(f"  Selected individuals per pool: {n_selected}")

## Create Founder Populations

Generate a founder population and split it into local and exotic founder groups.


In [ ]:
print("Creating founder haplotypes...")

founder_pop = runMacs(
    nInd=n_founders_per_pool * 2,
    nChr=n_chr,
    segSites=(n_qtl_per_trait * 2) + n_snp,
    inbred=False,
    species="CATTLE"
)

SP = SimParam(founder_pop)
SP.restrSegSites(minQtlPerChr=n_qtl_per_trait * 2, minSnpPerChr=n_snp)
SP.addSnpChip(n_snp)

# Trait 1: tick count incidence
SP.addTraitA(nQtlPerChr=n_qtl_per_trait, mean=0.0, var=1.0)

# Trait 2: body weight
SP.addTraitA(nQtlPerChr=n_qtl_per_trait, mean=0.0, var=1.0)

SP.setVarE(varE=[varE_tick, varE_weight])

local_founders = newPop(founder_pop[0:n_founders_per_pool], simParam=SP)
exotic_founders = newPop(founder_pop[n_founders_per_pool:(2 * n_founders_per_pool)], simParam=SP)

print(f"Local founders: {local_founders.n_ind}")
print(f"Exotic founders: {exotic_founders.n_ind}")
print("Founder populations created.")

## Advance Local and Exotic Populations to Generation 20

Both founder groups are advanced independently by random mating for 20 generations, matching the BRAID workflow.


In [ ]:
def advanceRandomMating(pop, nGenerations, nCrosses, simParam):
    current = pop
    history = []
    for generation in range(1, nGenerations + 1):
        current = randCross(current, nCrosses=nCrosses, simParam=simParam)
        history.append({
            "generation": generation,
            "meanG_trait1": meanG(current)[0],
            "meanG_trait2": meanG(current)[1],
            "varG_trait1": varG(current)[0],
            "varG_trait2": varG(current)[1],
            "n_ind": current.n_ind
        })
    return current, pd.DataFrame(history)

print("Advancing local population...")
local_population_g20, local_history = advanceRandomMating(
    local_founders, nGenerations=n_advance, nCrosses=n_crosses_per_generation, simParam=SP
)

print("Advancing exotic population...")
exotic_population_g20, exotic_history = advanceRandomMating(
    exotic_founders, nGenerations=n_advance, nCrosses=n_crosses_per_generation, simParam=SP
)

print("Advancement complete.")
print("Local generation 20 meanG:", meanG(local_population_g20))
print("Exotic generation 20 meanG:", meanG(exotic_population_g20))

## Apply Selection at Generation 20

The BRAID workflow specifies:
- **local population**: truncation selection on **phenotype** for tick count incidence
- **exotic population**: truncation selection on **breeding value** for body weight

For the local population, lower tick count is assumed favorable, so the best individuals are those with the lowest phenotypic values.

For the exotic population, the BRAID file specifies a placeholder genomic prediction model. In this notebook, true genetic value for body weight is used as a deterministic EBV proxy.


In [ ]:
# Phenotype local population for both traits
local_population_g20 = setPheno(local_population_g20, varE=[varE_tick, varE_weight], simParam=SP)

# Select local individuals with lowest phenotype for trait 1 (tick count incidence)
local_pheno_trait1 = np.array(local_population_g20.pheno)[:, 0]
local_best_idx = np.argsort(local_pheno_trait1)[:n_selected].tolist()
selected_local_population = selectInd(
    local_population_g20,
    nInd=n_selected,
    parents=local_best_idx,
    simParam=SP
)

# Evaluate exotic population and use breeding value proxy for trait 2 (body weight)
exotic_population_g20 = setPheno(exotic_population_g20, varE=[varE_tick, varE_weight], simParam=SP)
exotic_gv_trait2 = np.array(exotic_population_g20.gv)[:, 1]
exotic_best_idx = np.argsort(-exotic_gv_trait2)[:n_selected].tolist()
selected_exotic_population = selectInd(
    exotic_population_g20,
    nInd=n_selected,
    parents=exotic_best_idx,
    simParam=SP
)

print("Selection complete.")
print(f"Selected local population size: {selected_local_population.n_ind}")
print(f"Selected exotic population size: {selected_exotic_population.n_ind}")
print("Selected local meanG:", meanG(selected_local_population))
print("Selected exotic meanG:", meanG(selected_exotic_population))

## Produce Hybrids from Generations 21 to 40

The BRAID control section defines a repeated hybrid production phase from generations 21 to 40.  
Here, each generation is represented by crossing the selected local and exotic populations and tracking summary statistics.


In [ ]:
hybrid_results = []
current_local = selected_local_population
current_exotic = selected_exotic_population

for generation in range(21, 41):
    hybrid_pop = randCross(current_local, nCrosses=n_hybrid_crosses, parents=None, simParam=SP)
    hybrid_pop = setPheno(hybrid_pop, varE=[varE_tick, varE_weight], simParam=SP)

    hybrid_results.append({
        "generation": generation,
        "meanG_tick": meanG(hybrid_pop)[0],
        "meanG_weight": meanG(hybrid_pop)[1],
        "varG_tick": varG(hybrid_pop)[0],
        "varG_weight": varG(hybrid_pop)[1],
        "n_ind": hybrid_pop.n_ind
    })

hybrid_df = pd.DataFrame(hybrid_results)

print("Hybrid production complete.")
print(hybrid_df.head())

## Summarize Advancement and Hybrid Results

In [ ]:
local_history["population"] = "local"
exotic_history["population"] = "exotic"
advance_df = pd.concat([local_history, exotic_history], ignore_index=True)

print("Advancement summary:")
print(advance_df.head())

print("\nHybrid summary:")
print(hybrid_df.head())

## Plot Results

Visualize genetic means and variances across the advancement and hybrid phases.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for pop_name, df_sub in advance_df.groupby("population"):
    axes[0, 0].plot(df_sub["generation"], df_sub["meanG_trait1"], label=pop_name)
    axes[0, 1].plot(df_sub["generation"], df_sub["meanG_trait2"], label=pop_name)
    axes[1, 0].plot(df_sub["generation"], df_sub["varG_trait1"], label=pop_name)
    axes[1, 1].plot(df_sub["generation"], df_sub["varG_trait2"], label=pop_name)

axes[0, 0].set_title("Advancement meanG: tick count incidence")
axes[0, 1].set_title("Advancement meanG: body weight")
axes[1, 0].set_title("Advancement varG: tick count incidence")
axes[1, 1].set_title("Advancement varG: body weight")

for ax in axes.flatten():
    ax.set_xlabel("Generation")
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend()

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hybrid_df["generation"], hybrid_df["meanG_tick"], label="tick count incidence")
axes[0].plot(hybrid_df["generation"], hybrid_df["meanG_weight"], label="body weight")
axes[0].set_title("Hybrid mean genetic values")
axes[0].set_xlabel("Generation")
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend()

axes[1].plot(hybrid_df["generation"], hybrid_df["varG_tick"], label="tick count incidence")
axes[1].plot(hybrid_df["generation"], hybrid_df["varG_weight"], label="body weight")
axes[1].set_title("Hybrid genetic variances")
axes[1].set_xlabel("Generation")
axes[1].grid(True, linestyle="--", alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()

## Summary

This notebook translated the BRAID abstraction into an AlphaSimPy workflow with:

1. **Two founder populations** representing local and exotic cattle sources
2. **Independent advancement** by random mating from generation 0 to generation 20
3. **Phenotypic selection** in the local population for tick count incidence
4. **Genomic-selection-style selection** in the exotic population for body weight using an EBV proxy
5. **Repeated hybrid production** from generations 21 to 40
6. **Tracking of genetic mean and genetic variance** across the program

If more detailed operational information becomes available from the original breeding diagram, this notebook can be refined further with explicit mating designs, sex-specific roles, training populations, and formal genomic prediction models.
